In [9]:
from flax import nnx
import jax
import jax.numpy as jnp
import pandas as pd
import glob
from pathlib import Path
import json
import os
import sys
import shlex
from collections import defaultdict
sys.path.append("..")

from qnrl.algorithms.algorithm import Algorithm

import wandb
api = wandb.Api()

In [ ]:
root_sweeps = Path("../../wandb-sweep-data-qnrl/")
root_sweeps.mkdir(parents=True, exist_ok=True)

In [11]:
project_id = "qnrl"

In [13]:
sweep_name_id = { # name: id
    # QnRL.
    'qnrl/FrozenLake-v1': '7yoeyqqr',
    'qnrl/CliffWalking-v1': 'nin93kq8',
    'qnrl/Acrobot-v1': '5ehazde3',
    'qnrl/CartPole-v1': 'qojis1fw',

    # C51.
    'c51/FrozenLake-v1': 'jzduwzvg',
    'c51/CliffWalking-v1': '8xyt1eb5',
    'c51/Acrobot-v1': 'n8imxz1v',
    'c51/CartPole-v1': 'arv708xs',
    
    # Breakout-v5.
    'qnrl/Breakout-v5': 'eigl7wk5',
    'c51/Breakout-v5': 'dg4sodfc',
    
    # SpaceInvaders-v5.
    'qnrl/SpaceInvaders-v5': '8adr9d1k',
    'c51/SpaceInvaders-v5': 'wbmr06xp',
}
best_runs_for_sweep = { # name: [id]
    # QnRL.
    "qnrl/FrozenLake-v1": ['xy1qeozs', 'nrg58af6', 'ohbnwer4', '1w6f17u8'],
    "qnrl/CliffWalking-v1": ['84ly3kb8', 'q14zmjpp', 'ujlm00gt', 'er4d6rnz'],
    "qnrl/Acrobot-v1": ['k9tdwgod', '7iae54v0', 'n7t9tfkv', 'se6p60c1'],
    "qnrl/CartPole-v1": ['0jash5pr', '8pi89rle', 'zds1hi8i', 'tqeh89te'],
    
    # C51.
    "c51/FrozenLake-v1": ['907k571g', 'i8w40ma5', '8p7uzbpv', '8oqjux3k'],
    "c51/CliffWalking-v1": ['zko23iy1', 'd962vucv', 'azhg19ww', '0nn68l3j'],
    "c51/Acrobot-v1": ['zbzemhwh', 'hs1wv6fh', 'ec4ti73g', 'y18igax1'],
    "c51/CartPole-v1": ['ubxwaw3p', 'f20h31mw', 'gdixmx80', '9eihhl7r'],
    
    # Breakout-v5.
    'qnrl/Breakout-v5': ['i8ijgd1m', 'syrhxro1', '4pc6bthv', '3pv6knud'],
    'c51/Breakout-v5': ['bxdtzg0b', 'cmn3go73', '62raq6de', '2vxp36ul'],
    
    # SpaceInvaders-v5.
    'qnrl/SpaceInvaders-v5': ['kiwbzgdv', '4kkrr3v5', '2artbn9x', 'nvmd7j1x'],
    'c51/SpaceInvaders-v5': ['wq1meahd', 'gbkustul', 'q75ssoqq', 'ekmn2epv'],
}

In [14]:
import yaml
import ast
from qnrl.serialization import hash_serializable_object
from typing import Any

def get_run_config(config_file: Path):
    with open(str(config_file), 'r') as f:
        config = yaml.safe_load(f)
    return config

def get_run_metadata(metadata_file: Path):
    # Load metadata as JSON dictionary.
    with open(str(metadata_file), 'r') as f:
        metadata = json.load(f)
    return metadata

def get_run_args(metadata_file: Path) -> dict:
    metadata = get_run_metadata(metadata_file)
    l: list[tuple[str, str]] = [tokens.split('=', 1) for tokens in metadata['args'] if '--' in tokens and '=' in tokens]
    d = dict((key.replace('--', '', 1), ast.literal_eval(val)) for key, val in l)
    return d

def get_run_config_dict_from_metadata(metadata_file: Path) -> dict:
    """Compute run config hash from original run program command string."""
    arg_dict = get_run_args(metadata_file)
    return arg_dict['config']
    
    
    # metadata = get_run_metadata(metadata_file)
    
    # # Get config string from run program command arguments.
    # # Format is: --config="{...}"
    # cfg_str = metadata['args'][1].split("--config=")[1]
    # # Convert this string to dictionary via literal evaluation.
    # cfg_dict = ast.literal_eval(cfg_str)
    # return cfg_dict

In [15]:
from typing import Iterator

def eval(sweep_id: str, run_id: str, eval_kwargs_override: dict = {}) -> Iterator[dict]:
    
    dir_run = root_sweeps/f"sweep-{sweep_id}"/f"run-{run_id}"
    metadata_file = dir_run/"wandb-metadata.json"

    arg_dict = get_run_args(metadata_file)
    
    print(f"Evaluating: sweep={sweep_id}, run={run_id}, num_eval={arg_dict['num_eval']}")

    cfg = arg_dict['config']
    # cfg_hash, _ = hash_serializable_object(cfg) # Compute hash for config object.
    algo_config = Algorithm.from_dict(cfg)
    
    # Load parameters from model save.
    model_dirs = {Path(root).stem: Path(root) for root in glob.glob(str(dir_run/"models"/"*"))}
    algo_config.load(model_dirs)
    # nnx.display(nnx.state(algo_config.model))
    
    eval_reports = []
    eval_seed = arg_dict['eval_seed'] if arg_dict['eval_seed'] is not None else algo_config.seed
    key = jax.random.PRNGKey(seed=eval_seed)
    for eval_idx in range(arg_dict['num_eval']):
        
        # Generate a new random key.
        key, subkey = jax.random.split(key=key)
        
        # Evaluate the model using a different random seed.
        seed_eval = int(subkey[0])
        eval_kwargs = {}
        if 'eval_total_timesteps' in arg_dict and arg_dict['eval_total_timesteps'] is not None:
            eval_kwargs['total_timesteps'] = arg_dict['eval_total_timesteps']
        _, eval_report = algo_config.eval(seed=seed_eval, **{**eval_kwargs, **eval_kwargs_override})
        
        # Add an index counter for the evaluation report.
        eval_report['eval/index'] = eval_idx
        
        yield eval_report


def eval_via_env(env_key: str, eval_kwargs_override: dict = {}):
    eval_reports = defaultdict(lambda: defaultdict(list[dict])) # { sweep_id: { run_id: [report_0, report_1, ...] } }
    sweep_id = sweep_name_id[env_key]
    for run_id in best_runs_for_sweep[env_key]:
        eval_reports[sweep_id][run_id] = list(eval(sweep_id, run_id, eval_kwargs_override))
    return eval_reports

In [20]:
# env_key = "qnrl/CartPole-v1"
# env_key = "c51/CartPole-v1"

# env_key = "qnrl/FrozenLake-v1"
# env_key = "c51/FrozenLake-v1"

# env_key = "qnrl/Acrobot-v1"
# env_key = "c51/Acrobot-v1"

# env_key = "qnrl/SpaceInvaders-v5"
# env_key = "c51/SpaceInvaders-v5"

env_key = "qnrl/Breakout-v5"
# env_key = "c51/Breakout-v5"

# eps = 0.05 # What Bellemare used for training.
# eps = 0.001 # What Bellemare used for eval.
eps = 0.0
report_dict = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))

Evaluating: sweep=eigl7wk5, run=i8ijgd1m, num_eval=10


 33%|███▎      | 326/1000 [00:07<00:15, 43.49step/s, eval/step=326, eval/episode_reward=12, eval/episode_length=327, eval/episode_time=7.48]


Evaluating: sweep=eigl7wk5, run=syrhxro1, num_eval=10


 15%|█▌        | 150/1000 [00:03<00:19, 43.17step/s, eval/step=150, eval/episode_reward=4, eval/episode_length=151, eval/episode_time=3.46]


Evaluating: sweep=eigl7wk5, run=4pc6bthv, num_eval=10


 17%|█▋        | 173/1000 [00:04<00:19, 43.07step/s, eval/step=173, eval/episode_reward=5, eval/episode_length=174, eval/episode_time=4]


Evaluating: sweep=eigl7wk5, run=3pv6knud, num_eval=10


 16%|█▌        | 157/1000 [00:03<00:19, 43.55step/s, eval/step=157, eval/episode_reward=4, eval/episode_length=158, eval/episode_time=3.59]


In [21]:
nnx.display(report_dict)

In [22]:
def get_metric_as_series_for_run(metric_key: str, run_id: str, eval_reports: list[dict]):
    """Extract metric key as Pandas series and change metric key to row ID."""
    return pd.DataFrame(eval_reports)[metric_key].rename(run_id)

def get_metric_as_dataframe_for_sweep(metric_key: str, sweep_id: str):
    l = []
    for run_id, eval_reports in report_dict[sweep_id].items():
        # l.append(pd.DataFrame(eval_reports)['eval/episode_reward'].to_list())
        l.append(pd.DataFrame(eval_reports)[metric_key].rename(run_id))
    df = pd.concat(l, axis=1, join='outer') # column keys are 'row_id'.
    return df

In [23]:
for sweep_id in report_dict.keys():
    df = get_metric_as_dataframe_for_sweep('eval/episode_reward', sweep_id)
    print(df)
    print(df.mean())
    print(df.mean().mean())

   i8ijgd1m  syrhxro1  4pc6bthv  3pv6knud
0       2.0       1.0       4.0       1.0
1       2.0       9.0       2.0       2.0
2       2.0       9.0       2.0       2.0
3      12.0       4.0       5.0       4.0
4       2.0       1.0       4.0       1.0
5      12.0       4.0       5.0       4.0
6      12.0       4.0       5.0       4.0
7      12.0       4.0       5.0       4.0
8      12.0       4.0       5.0       4.0
9      12.0       4.0       5.0       4.0
i8ijgd1m    8.0
syrhxro1    4.4
4pc6bthv    4.2
3pv6knud    3.0
dtype: float64
4.9
